In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
from openai import OpenAI
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
print(len(documents))

72


In [5]:
from minsearch import Index

def build_index(documents):
	index = Index(
		text_fields=["content"],
		keyword_fields=["filename"]
	)
	index.fit(documents)
	return index

index = build_index(documents)
query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query)

In [6]:
print(results[0]['filename'])

01-agentic-rag/lessons/14-agentic-loop.md


In [7]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-21 19:55:55--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-21 19:55:55 (22.0 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]



In [8]:
from rag_helper import RAGBase

assistant = RAGBase(
	index=index,
	llm_client=openai_client
)

In [10]:
query = "How does the agentic loop keep calling the model until it stops?"

answer, used_tokens = assistant.rag(query)

print("Answer:", answer)
print("Used input tokens:", used_tokens)

Answer: The agentic loop keeps calling the model until it stops by using a `while` loop that continues to prompt the model until it returns a response without any function calls. 

Here's how it works:

1. The loop starts by sending a message to the model and getting a response.
2. If the response contains a function call, the loop executes the function call and appends the result to the conversation history.
3. The loop then sends the updated conversation history back to the model and gets a new response.
4. Steps 2 and 3 repeat until the model returns a response without any function calls.
5. At this point, the loop stops, and the final response from the model is returned as the answer.

The loop uses a flag `has_function_calls` to track whether the model's response contains any function calls. If the response does not contain any function calls, the flag is set to `False`, and the loop exits.

Here's the relevant code snippet:

```python
it = 1

while True:
    print(f"iteration #{i

In [11]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


In [12]:
chunk_index = build_index(chunks)

assistant_chunk = RAGBase(
	index=chunk_index,
	llm_client=openai_client
)

query = "How does the agentic loop keep calling the model until it stops?"

answer, used_tokens = assistant_chunk.rag(query)

print("Used input tokens:", used_tokens)

Used input tokens: 2311


In [13]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

def search(query: str) -> dict[str, str]:
    """
    Search the course lessons database for entries matching the given query.
    """
    return chunk_index.search(query, num_results=5)

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(
        model='llama-3.3-70b-versatile',
        client=openai_client
    )
)

In [ ]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

result.all_messages